# Tutorial 2: Composition

This tutorial builds on [Tutorial 1 (counter)](counter.ipynb). We define an inverted pendulum environment and a controller neural network, then **compose** them into a closed-loop system using shared wires.

You'll learn:
1. How modules get their own fresh wires by default
2. How to create shared wire pairs and pass them to `Wrapper()` / `Module()` wrappers
3. How `Wrapper` composes modules that share wires

## The inverted pendulum environment

An inverted pendulum balanced upright. The full nonlinear dynamics:

$$\ddot\theta = \frac{g}{\ell}\,\sin\theta + \frac{\tau}{m\ell^2}$$

The positive gravitational term makes the upright equilibrium ($\theta = 0$) **unstable** — without active control, any perturbation grows. The controller must apply torque $\tau$ to keep it balanced.

Write a standard `gym.Env`: `__init__` sets spaces and state, `reset` initializes, `step` updates. Then wrap with `Wrapper()` to extract the symbolic module.

In [1]:
import math
import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
from zrth.gym import Wrapper
from zrth.torch import Module


class InvertedPendulumEnv(gym.Env):
    """Inverted pendulum with full nonlinear dynamics."""

    def __init__(self, g=9.81, l=1.0, m=1.0, dt=0.05):
        super().__init__()

        self.action_space = spaces.Box(low=-2.0, high=2.0, shape=(1,))
        self.observation_space = spaces.Box(low=-10.0, high=10.0, shape=(1,))

        self.g = g
        self.l = l
        self.m = m
        self.dt = dt

        # Also set state variables here so the analyzer can infer their dtype
        self.theta = 0.02
        self.theta_dot = 0.0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.theta = 0.02
        self.theta_dot = 0.0

        observation = self.theta
        return observation, {}

    def step(self, torque):
        accel = (self.g / self.l) * math.sin(self.theta) + torque / (self.m * self.l * self.l)
        self.theta_dot = self.theta_dot + accel * self.dt
        self.theta = self.theta + self.theta_dot * self.dt

        observation = self.theta
        reward = 0.0 - self.theta * self.theta - 0.1 * self.theta_dot * self.theta_dot
        terminated = self.theta > 3.14 or self.theta < -3.14
        truncated = False
        return observation, reward, terminated, truncated, {}

## Wrapping and composition

We create shared wire pairs up front so that the pendulum's observation feeds the controller's input, and the controller's output feeds the pendulum's action. Then we wrap both and compose them into a closed-loop system.

In [2]:
from zrth import Wire, Float

pendulum = InvertedPendulumEnv(g=9.81, l=1.0, m=1.0, dt=0.05)

# Shared wire pairs: pendulum observation = controller input
obs_wire = [Wire(Float(1)), Wire(Float(1))]
# Shared wire pairs: controller output = pendulum action
act_wire = [Wire(Float(1)), Wire(Float(1))]

wrapped_pendulum = Wrapper(pendulum, observation=obs_wire, action=act_wire)
print(wrapped_pendulum)

module
  external
    w2, w3 : Float(1)
  interface
    w0, w1 : Float(1)
    w12, w13 : Float(1)
    w14, w15 : Bool(1)
    w16, w17 : Bool(1)
  private
    w4, w5 : Float(1)
    w6, w7 : Float(1)
  atom controls w1, w5, w7, w13, w15, w17 reads w2, w4, w6
  init
    Tensor([0.05000000074505806]) w8; 
    Tensor([1]) w9; 
    Tensor([1]) w10; 
    Tensor([9.8100004196167]) w11; 
    Tensor([0.019999999552965164]) w18; 
    Id w5; w18
    Tensor([0]) w19; 
    Id w7; w19
    Id w1; w18
    Tensor([0]) w13; 
    Const: false w15; 
    Const: false w17; 
  update
    Tensor([0.05000000074505806]) w8; 
    Tensor([1]) w9; 
    Tensor([1]) w10; 
    Tensor([9.8100004196167]) w11; 
    Div w20; w11, w9
    Sin w21; w4
    Mul w22; w20, w21
    Mul w23; w10, w9
    Mul w24; w23, w9
    Div w25; w2, w24
    Add w26; w22, w25
    Mul w27; w26, w8
    Add w28; w6, w27
    Id w7; w28
    Mul w29; w28, w8
    Add w30; w4, w29
    Id w5; w30
    Tensor([0]) w31; 
    Mul w32; w30, w30
    Sub w33; 

A small NN that observes $\theta$ and outputs a torque $\tau$. Architecture: $[1] \to 16 \to [1]$. We wrap it with the same shared wires.

In [3]:
class ControllerNN(nn.Module):
    """NN controller: theta -> torque. Uses tanh for symmetric output."""

    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(1, 16)
        self.fc2 = nn.Linear(16, 1)

    def forward(self, state):
        x = torch.tanh(self.fc1(state))
        return 2.0 * torch.tanh(self.fc2(x))  # scale to [-2, 2]

In [4]:
controller = ControllerNN()
wrapped_controller = Module(controller, extl=obs_wire, intf=act_wire)
print(wrapped_controller)

module
  external
    w0, w1 : Float(1)
  interface
    w2, w3 : Float(1)
  atom controls w3 awaits w1
  init
    Tensor([0.8359413146972656 -0.7336455583572388 0.4205350875854492 -0.6647641658782959 -0.3362523317337036 ...]) w48; 
    Tensor([-0.3498821258544922 -0.12429368495941162 -0.8933138847351074 -0.022783756256103516 -0.7534104585647583 ...]) w49; 
    Linear w50; w1, w48, w49
    Tanh w51; w50
    Tensor([2]) w52; 
    Tensor([0.21008434891700745 -0.13836601376533508 -0.19865190982818604 0.1205480694770813 0.24379825592041016 ...]) w53; 
    Tensor([-0.17422360181808472]) w54; 
    Linear w55; w51, w53, w54
    Tanh w56; w55
    Mul w57; w52, w56
    Id w3; w57
  update
    Tensor([0.8359413146972656 -0.7336455583572388 0.4205350875854492 -0.6647641658782959 -0.3362523317337036 ...]) w48; 
    Tensor([-0.3498821258544922 -0.12429368495941162 -0.8933138847351074 -0.022783756256103516 -0.7534104585647583 ...]) w49; 
    Linear w50; w1, w48, w49
    Tanh w51; w50
    Tensor([2]) 

In [5]:
closed_loop = Wrapper(wrapped_pendulum, wrapped_controller)
print(closed_loop)

module
  interface
    w0, w1 : Float(1)
    w12, w13 : Float(1)
    w14, w15 : Bool(1)
    w16, w17 : Bool(1)
    w2, w3 : Float(1)
  private
    w4, w5 : Float(1)
    w6, w7 : Float(1)
  atom controls w1, w5, w7, w13, w15, w17 reads w2, w4, w6
  init
    Tensor([0.05000000074505806]) w8; 
    Tensor([1]) w9; 
    Tensor([1]) w10; 
    Tensor([9.8100004196167]) w11; 
    Tensor([0.019999999552965164]) w18; 
    Id w5; w18
    Tensor([0]) w19; 
    Id w7; w19
    Id w1; w18
    Tensor([0]) w13; 
    Const: false w15; 
    Const: false w17; 
  update
    Tensor([0.05000000074505806]) w8; 
    Tensor([1]) w9; 
    Tensor([1]) w10; 
    Tensor([9.8100004196167]) w11; 
    Div w20; w11, w9
    Sin w21; w4
    Mul w22; w20, w21
    Mul w23; w10, w9
    Mul w24; w23, w9
    Div w25; w2, w24
    Add w26; w22, w25
    Mul w27; w26, w8
    Add w28; w6, w27
    Id w7; w28
    Mul w29; w28, w8
    Add w30; w4, w29
    Id w5; w30
    Tensor([0]) w31; 
    Mul w32; w30, w30
    Sub w33; w31, w32
  

In [6]:
from zrth.visual import show
stop = show(closed_loop)

Module viewer running at http://127.0.0.1:61609 (ws://127.0.0.1:61610)


## Training the controller

We train the controller using **REINFORCE** (policy gradient). The wrapped objects are fully functional — `wrapped_pendulum.reset()` / `wrapped_pendulum.step()` work as gym methods, and `wrapped_controller(obs)` runs a PyTorch forward pass with autograd.

The controller outputs a mean torque $\mu$; we sample from $\mathcal{N}(\mu, \sigma^2)$ to explore. The loss is the standard REINFORCE estimator:

$$\nabla J = \mathbb{E}\left[\sum_t \nabla \log \pi(a_t | s_t) \cdot G_t\right]$$

where $G_t$ is the discounted return from step $t$.

Because `zrth.torch.Module` uses live tensor references, the symbolic module and the composed `closed_loop` automatically reflect the trained weights — no re-wrapping needed.

In [7]:
def compute_returns(rewards, gamma=0.99):
    """Compute discounted returns for each timestep."""
    returns = []
    G = 0.0
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return returns

optimizer = torch.optim.Adam(wrapped_controller.parameters(), lr=0.005)
n_episodes = 1000
max_steps = 200
gamma = 0.99

for episode in range(n_episodes):
    wrapped_pendulum.reset()
    log_probs = []
    rewards = []

    sigma = max(0.3, 1.0 - episode / 500)  # decay exploration noise

    for step in range(max_steps):
        state = torch.tensor([wrapped_pendulum.theta], dtype=torch.float32)
        mu = wrapped_controller(state.unsqueeze(0)).squeeze()
        dist = torch.distributions.Normal(mu, sigma)
        action = dist.sample().clamp(-2.0, 2.0)
        log_probs.append(dist.log_prob(action))

        _, reward, terminated, _, _ = wrapped_pendulum.step(action.item())
        rewards.append(reward + 0.1)  # survival bonus
        if terminated:
            break

    # Policy gradient update
    returns = compute_returns(rewards, gamma)
    returns = torch.tensor(returns, dtype=torch.float32)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)  # baseline

    loss = torch.tensor(0.0)
    for lp, G in zip(log_probs, returns):
        loss = loss - lp * G

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if episode % 100 == 0:
        total_reward = sum(rewards)
        print(f"episode {episode:3d}  steps = {len(rewards):3d}  reward = {total_reward:.2f}  sigma = {sigma:.2f}")

episode   0  steps =  34  reward = -72.48  sigma = 1.00
episode 100  steps =  31  reward = -68.32  sigma = 0.80
episode 200  steps =  44  reward = -72.53  sigma = 0.60
episode 300  steps =  37  reward = -67.90  sigma = 0.40
episode 400  steps =  37  reward = -66.71  sigma = 0.30
episode 500  steps =  36  reward = -68.40  sigma = 0.30
episode 600  steps =  62  reward = -77.38  sigma = 0.30
episode 700  steps =  29  reward = -68.59  sigma = 0.30
episode 800  steps =  31  reward = -73.83  sigma = 0.30
episode 900  steps =  34  reward = -80.34  sigma = 0.30


## Evaluating the trained controller

Run the trained controller deterministically (no exploration noise) and print $\theta$ at each step.

In [8]:
wrapped_pendulum.reset()

print(f"{'step':>4}  {'theta':>8}  {'theta_dot':>10}  {'torque':>8}  {'reward':>8}")
print("-" * 50)

with torch.no_grad():
    for step in range(20):
        state = torch.tensor([wrapped_pendulum.theta], dtype=torch.float32)
        torque = wrapped_controller(state.unsqueeze(0)).squeeze().item()
        torque = max(-2.0, min(2.0, torque))

        _, reward, terminated, _, _ = wrapped_pendulum.step(torque)
        print(f"{step:4d}  {wrapped_pendulum.theta:8.4f}  {wrapped_pendulum.theta_dot:10.4f}  {torque:8.4f}  {reward:.4f}")
        if terminated:
            print("terminated!")
            break

step     theta   theta_dot    torque    reward
--------------------------------------------------
   0    0.0200     -0.0010   -0.2152  -0.0004
   1    0.0199     -0.0019   -0.2152  -0.0004
   2    0.0197     -0.0029   -0.2152  -0.0004
   3    0.0195     -0.0040   -0.2151  -0.0004
   4    0.0192     -0.0052   -0.2151  -0.0004
   5    0.0189     -0.0065   -0.2150  -0.0004
   6    0.0185     -0.0080   -0.2149  -0.0003
   7    0.0180     -0.0097   -0.2148  -0.0003
   8    0.0175     -0.0115   -0.2146  -0.0003
   9    0.0168     -0.0137   -0.2144  -0.0003
  10    0.0160     -0.0162   -0.2142  -0.0003
  11    0.0150     -0.0191   -0.2139  -0.0003
  12    0.0139     -0.0224   -0.2136  -0.0002
  13    0.0126     -0.0262   -0.2133  -0.0002
  14    0.0110     -0.0307   -0.2129  -0.0002
  15    0.0093     -0.0359   -0.2124  -0.0002
  16    0.0072     -0.0419   -0.2118  -0.0002
  17    0.0047     -0.0490   -0.2111  -0.0003
  18    0.0018     -0.0572   -0.2103  -0.0003
  19   -0.0015     -0.0668  

## Running the composed closed-loop system

The `closed_loop` is a `Wrapper` — it composes the pendulum and controller into a single Module, while keeping the real pendulum env for delegation. When we call `step()`, the pendulum atom runs the real env (for rendering, real physics) and the controller atom runs symbolically. Outputs flow between them via shared wires.

Because the symbolic module holds live tensor references to the controller's weights, the composed module already reflects the trained values — no re-wrapping needed.

In [9]:
closed_loop.reset()

print(f"{'step':>4}  {'theta':>8}  {'theta_dot':>10}")
print("-" * 30)

for step in range(20):
    obs, reward, terminated, truncated, info = closed_loop.step(0)
    print(f"{step:4d}  {closed_loop.theta:8.4f}  {closed_loop.theta_dot:10.4f}")
    if terminated:
        print("terminated!")
        break

step     theta   theta_dot
------------------------------
   0    0.0205      0.0098
   1    0.0215      0.0199
   2    0.0230      0.0304
   3    0.0251      0.0417
   4    0.0278      0.0540
   5    0.0312      0.0676
   6    0.0353      0.0829
   7    0.0403      0.1002
   8    0.0463      0.1200
   9    0.0535      0.1427
  10    0.0619      0.1689
  11    0.0719      0.1992
  12    0.0836      0.2345
  13    0.0974      0.2754
  14    0.1135      0.3231
  15    0.1324      0.3787
  16    0.1546      0.4434
  17    0.1806      0.5190
  18    0.2109      0.6071
  19    0.2464      0.7097


## Verifying execution order and correctness

The composed module has two atoms. The topological sort orders them by **await dependencies**: the controller `awaits` the pendulum's observation wire, so the pendulum atom executes first. We can inspect this directly.

We also validate that `Wrapper` (which runs the real pendulum env for the env atom and the symbolic interpreter for the controller atom) produces identical results to `Env` (which runs both atoms purely symbolically).

In [10]:
# Execution order: inspect the atoms and their await dependencies
for idx in range(len(closed_loop.atoms)):
    atom = closed_loop.atoms[idx]
    n_ctrl = len(atom.ctrl)
    n_wait = len(atom.wait)
    ctrl_ids = [atom.ctrl[j].id for j in range(n_ctrl)]
    wait_ids = [atom.wait[j].id for j in range(n_wait)]

    is_env = idx == closed_loop._env_atom_idx
    label = "pendulum (real env)" if is_env else "controller (symbolic)"

    print(f"Atom {idx}: {label}")
    print(f"  controls: {ctrl_ids}")
    print(f"  awaits:   {wait_ids}")
    print()

Atom 0: pendulum (real env)
  controls: [1, 5, 7, 13, 15, 17]
  awaits:   []

Atom 1: controller (symbolic)
  controls: [3]
  awaits:   [1]



In [11]:
import numpy as np
from zrth.gym import Env

# Compare Wrapper (real env + symbolic controller) vs Env (pure symbolic)
sim_closed = Env(wrapped_pendulum, wrapped_controller)

closed_loop.reset()
sim_closed.reset()

print(f"{'step':>4}  {'Wrapper theta':>14}  {'Env theta':>12}  {'match':>8}")
print("-" * 50)

all_match = True
for step in range(20):
    closed_loop.step(0)
    sim_closed.step(np.zeros(1))

    env_theta = closed_loop.theta
    sim_theta = float(sim_closed._state[sim_closed._pairs['observation'][0]].item())
    match = abs(env_theta - sim_theta) < 1e-6
    all_match = all_match and match
    print(f"{step:4d}  {env_theta:14.6f}  {sim_theta:12.6f}  {'OK' if match else 'MISMATCH':>8}")

print()
print(f"All steps match: {all_match}")

step   Wrapper theta     Env theta     match
--------------------------------------------------
   0        0.020490      0.020490        OK
   1        0.021483      0.021483        OK
   2        0.023003      0.023003        OK
   3        0.025087      0.025087        OK
   4        0.027786      0.027786        OK
   5        0.031167      0.031167        OK
   6        0.035311      0.035311        OK
   7        0.040322      0.040322        OK
   8        0.046321      0.046321        OK
   9        0.053456      0.053456        OK
  10        0.061901      0.061901        OK
  11        0.071863      0.071863        OK
  12        0.083587      0.083587        OK
  13        0.097358      0.097358        OK
  14        0.113512      0.113512        OK
  15        0.132445      0.132445        OK
  16        0.154616      0.154616        OK
  17        0.180565      0.180565        OK
  18        0.210917      0.210917        OK
  19        0.246404      0.246404        OK

All